In [1]:
import json
import os
import sys
import pandas as pd
from pprint import pprint
from tqdm import tqdm
from langchain import PromptTemplate, FewShotPromptTemplate

sys.path.insert(0, '../src/')
from prompts import (
DEPRESSION_FEWSHOT_LANGCHAIN,
ANXIETY_FEWSHOT_LANGCHAIN,
COMORBIDITY_FEWSHOT_LANGCHAIN,
DEPRESSION_FEWSHOT_LLAMA_LANGCHAIN,
ANXIETY_FEWSHOT_LLAMA_LANGCHAIN,
COMORBIDITY_FEWSHOT_LLAMA_LANGCHAIN
)

In [2]:
test_path = '../data/test/full_test.csv'
corpus_path = '../data/silver_data/silver_labels_gpt_3.5_turbo.csv'
mapping_path_depression = "../data/mappings/semantic-similarity-depression.json"
mapping_path_anxiety = "../data/mappings/semantic-similarity-anxiety.json"
mapping_path_comorbid = "../data/mappings/semantic-similarity-comorbid.json"
mapping_path_normal = "../data/mappings/semantic-similarity-normal.json"


df_test = pd.read_csv(test_path).reset_index()
df_corpus = pd.read_csv(corpus_path).reset_index()
mapping_depression = json.load(open(mapping_path_depression))
mapping_anxiety = json.load(open(mapping_path_anxiety))
mapping_comorbid = json.load(open(mapping_path_comorbid))
mapping_normal = json.load(open(mapping_path_normal))


print('-'*50)
print(f"Test set size: {df_test.shape[0]}")
print(f"Corpus (Silver Labels) size: {df_corpus.shape[0]}")
print(f"Total mapping (Depression): {len(mapping_depression)}")
print(f"Total mapping (Anxiety): {len(mapping_anxiety)}")
print(f"Total mapping (Comorbid): {len(mapping_comorbid)}")
print(f"Total mapping (Normal): {len(mapping_normal)}")
print('-'*50)

--------------------------------------------------
Test set size: 2872
Corpus (Silver Labels) size: 7667
Total mapping (Depression): 2872
Total mapping (Anxiety): 2872
Total mapping (Comorbid): 2872
Total mapping (Normal): 2872
--------------------------------------------------


In [3]:
def merge_dicts(dict1, dict2):
    merged_dict = {}
    for i in dict1.keys():
        merged_dict[i] = dict1[i]
    for i in dict2.keys():
        merged_dict[i] = dict2[i]
    return merged_dict

df_corpus['depression_label'] = df_corpus['depression_label'].apply(lambda label: {"depression": "yes"} if label == 1 else {"depression": "no"})
df_corpus['anxiety_label'] = df_corpus['anxiety_label'].apply(lambda label: {"anxiety": "yes"} if label == 1 else {"anxiety": "no"})
df_corpus['comorbidity_label'] = df_corpus.apply(lambda row: merge_dicts(row['depression_label'],row['anxiety_label']), axis=1)
df_corpus['comorbidity_label'].value_counts()

comorbidity_label
{'depression': 'no', 'anxiety': 'no'}      3349
{'depression': 'yes', 'anxiety': 'no'}     2705
{'depression': 'no', 'anxiety': 'yes'}     1048
{'depression': 'yes', 'anxiety': 'yes'}     565
Name: count, dtype: int64

In [7]:
# Generate Few Shot prompts for each data point in test set.
topk = 2 # Provide only even number of examples, 2, 4, 6, 8, ..
text_col = 'text'
id_col = 'id'
task_type = 'depression' #(depression | anxiety | comorbidity)

prompts = []
for i in tqdm(range(df_test.shape[0]), desc=f"Generating few shot prompts"):
    few_shot_examples = []
    input_post = df_test.iloc[i][text_col]
    
    if task_type == 'depression':        
        exemplars_d = mapping_depression[df_test.iloc[i][id_col]]
        exemplar_ids_d = [i[0] for i in exemplars_d][:int(topk/2)]
        exemplars_n = mapping_normal[df_test.iloc[i][id_col]]
        exemplar_ids_n = [i[0] for i in exemplars_n][:int(topk/2)]
        exemplar_ids = exemplar_ids_d + exemplar_ids_n
 
        exemplar_df = df_corpus[df_corpus[id_col].isin(exemplar_ids)]
        exemplar_df = exemplar_df.sample(exemplar_df.shape[0])
        
        for j, row in exemplar_df.iterrows():
            exemplar_post = row['text']
            exemplar_label = row['depression_label']
            
            few_shot_examples.append(
                {
                    'post': exemplar_post,
                    'label': exemplar_label,
                }
            )
            
        prompt_template = DEPRESSION_FEWSHOT_LLAMA_LANGCHAIN['prompt_template']
        few_shot_prefix = DEPRESSION_FEWSHOT_LLAMA_LANGCHAIN['few_shot_prefix']
        few_shot_suffix = DEPRESSION_FEWSHOT_LLAMA_LANGCHAIN['few_shot_suffix'](input_post)
        
    
    elif task_type == 'anxiety':
        exemplars_a = mapping_anxiety[df_test.iloc[i][id_col]]
        exemplar_ids_a = [i[0] for i in exemplars_a][:int(topk/2)]
        exemplars_n = mapping_normal[df_test.iloc[i][id_col]]
        exemplar_ids_n = [i[0] for i in exemplars_n][:int(topk/2)]
        exemplar_ids = exemplar_ids_a + exemplar_ids_n

        exemplar_df = df_corpus[df_corpus[id_col].isin(exemplar_ids)]
        exemplar_df = exemplar_df.sample(exemplar_df.shape[0])

        for j, row in exemplar_df.iterrows():
            exemplar_post = row['text']
            exemplar_label = row['anxiety_label']
            
            few_shot_examples.append(
                {
                    'post': exemplar_post,
                    'label': exemplar_label,
                }
            )
            
        prompt_template = ANXIETY_FEWSHOT_LLAMA_LANGCHAIN['prompt_template']
        few_shot_prefix = ANXIETY_FEWSHOT_LLAMA_LANGCHAIN['few_shot_prefix']
        few_shot_suffix = ANXIETY_FEWSHOT_LLAMA_LANGCHAIN['few_shot_suffix'](input_post)
        
        
    elif task_type == 'comorbidity':
        if topk < 4: 
            topk = 4

        exemplars_d = mapping_depression[df_test.iloc[i][id_col]]
        exemplar_ids_d = [i[0] for i in exemplars_d][:int(topk/4)]
        exemplars_a = mapping_anxiety[df_test.iloc[i][id_col]]
        exemplar_ids_a = [i[0] for i in exemplars_a][:int(topk/4)]
        exemplars_c = mapping_comorbid[df_test.iloc[i][id_col]]
        exemplar_ids_c = [i[0] for i in exemplars_c][:int(topk/4)]
        exemplars_n = mapping_normal[df_test.iloc[i][id_col]]
        exemplar_ids_n = [i[0] for i in exemplars_n][:int(topk/4)]
        exemplar_ids = exemplar_ids_d + exemplar_ids_a + exemplar_ids_c + exemplar_ids_n

        exemplar_df = df_corpus[df_corpus[id_col].isin(exemplar_ids)]
        exemplar_df = exemplar_df.sample(exemplar_df.shape[0])

        for j, row in exemplar_df.iterrows():
            exemplar_post = row['text']
            exemplar_label = row['comorbidity_label']
            
            few_shot_examples.append(
                {
                    'post': exemplar_post,
                    'label': exemplar_label,
                }
            )
            
        prompt_template = COMORBIDITY_FEWSHOT_LLAMA_LANGCHAIN['prompt_template']
        few_shot_prefix = COMORBIDITY_FEWSHOT_LLAMA_LANGCHAIN['few_shot_prefix']
        few_shot_suffix = COMORBIDITY_FEWSHOT_LLAMA_LANGCHAIN['few_shot_suffix'](input_post)
               
        
    few_shot_examples = ''.join(prompt_template(x['post'], x['label']) for x in few_shot_examples)
    few_shot_prompt = ''.join([few_shot_prefix, few_shot_examples, few_shot_suffix])        
    prompts.append(few_shot_prompt)


df_test[f'few_shot_prompt_{task_type}'] = prompts
df_test

Generating few shot prompts: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2872/2872 [00:02<00:00, 1197.89it/s]


,index,id,title,selftext,multilabel_clf_label,disorder,text,multiclass_clf_label,depression_label,anxiety_label,few_shot_prompt_depression
0,0,8es6qn,I cut myself for the first time in a year toda...,... and hated that I still loved it. The burni...,"[1, 0]",{'depressive_disorder'},I cut myself for the first time in a year toda...,Depression,1,0,\nYou will be presented with a post and an ass...
1,1,ar89pm,Anybody hate spring/summer?,".....And even so, when depression is not too s...","[1, 1]","{'depressive_disorder', 'anxiety_disorder'}","Anybody hate springsummer. And even so, when d...",Comorbid (Depression + Anxiety),1,1,\nYou will be presented with a post and an ass...
2,2,5g4v1n,So I'm staring at this tablet of Lexapro...,...and I'm not sure what to do. I guess I don'...,"[1, 1]","{'depressive_disorder', 'anxiety_disorder'}",So Im staring at this tablet of Lexapro. and I...,Comorbid (Depression + Anxiety),1,1,\nYou will be presented with a post and an ass...
3,3,9x8h3g,"Found a poem sort of thing, very emo (ha ha ha...","""I am delicate and bitter. I am sweet on the o...","[1, 0]",{'depressive_disorder'},"Found a poem sort of thing, very emo ha ha ha ...",Depression,1,0,\nYou will be presented with a post and an ass...
4,4,es440i,You don't get it,"""It gets better"" ""Stop thinking about it"" ""get...","[1, 1]","{'depressive_disorder', 'anxiety_disorder'}",You dont get it. It gets better Stop thinking ...,Comorbid (Depression + Anxiety),1,1,\nYou will be presented with a post and an ass...
...,...,...,...,...,...,...,...,...,...,...,...
2867,2867,lm7uix,My life just feels like this one big party Im ...,You know the feeling. Where youre just so over...,"[1, 0]",{'depressive_disorder'},My life just feels like this one big party Im ...,Depression,1,0,\nYou will be presented with a post and an ass...
2868,2868,ac1aj9,Just move away from your parents!,You probably want to hear my sob story as much...,"[0, 0]",{'control_group'},Just move away from your parents. You probably...,Normal,0,0,\nYou will be presented with a post and an ass...
2869,2869,hng1f5,That feeling when...,"You wake up, and that itself just brings you s...","[0, 0]",{'control_group'},"That feeling when. You wake up, and that itsel...",Normal,0,0,\nYou will be presented with a post and an ass...
2870,2870,4pfk9x,Life just keeps getting more complicated,"You would think after years of bad breaks, som...","[0, 0]",{'control_group'},Life just keeps getting more complicated. You ...,Normal,0,0,\nYou will be presented with a post and an ass...


In [9]:
print(df_test.at[0, f'few_shot_prompt_{task_type}'])


You will be presented with a post and an assigned label to identify whether the poster shows symptoms of depression. Here are two examples:

Post: i fucking hate myself. i want to become a surgeon but at this rate, thats no where near possible. i just scored a 53 on my biology final and i was so close to a b and was actually kind of confident. now im going to have a c on my semester final. i studied hard for this test but im just a fucking idiot. someone also crashed their car into mine and took advantage of my age with the whole insurance process so thats fucking great.
Response: {'depression': 'no'}

Post: so, ive been inactive on this account for a while, but im here to just say stuff again like i did in my other two posts or whatever. it seems like ive just started doing it because i got bored. it gives me something to focus on instead of having to think about everything happening at school and home. its also a stress relief for me. whenever im pissed or stressed ill make a scratc

In [9]:
df1 = pd.read_csv("../data/few_shot_prompts/depression_num_examples_ss_4.csv")
df2 = pd.read_csv("../data/few_shot_prompts/mental_flan_t5_depression_ss_only_num_examples_per_label_2.csv")

In [10]:
print(df1.at[0, "few_shot_prompt_depression"])


Below are posts and their respective assessments based on the criteria for clinical depression as defined in the DSM-5.
Format your response as a JSON object {'depression': ''} with values either 'yes' or 'no'.

Post: before anyone asks, yes, i am diagnosed with depression. ive had it for 7 years but was only diagnosed a few weeks ago. i used to cut to let the pain out. then that stopped helping so it was just to feel something. now i dont even feel anything when i do it. i just do it because i can. and idk why. i feel nuts and im just waiting to get thrown in a hospital.
Assesement: {'depression': 'yes'}

Post: i self harmed tonight more times than i can count. i work in the medical field and im always trying to come up with solutions on how to end it all. all the pain and heartache. im 30 years old and nothing is getting better. i self harm because i feel like i deserve it. the scars on the outside some how feel as if they belong there because of how fucked up i feel on the inside. 

In [13]:
print(df2.at[7, "few_shot_prompt_depression"])

Below are posts and their respective assessments based on the criteria for clinical depression as defined in the DSM-5.
Format your response as a JSON object 'depression': '' with values either 'yes' or 'no'.

Post: i feel numb. i feel like a dark cloud is looming over me. most days i cant get out of bed, i go days and sometimes weeks on end where i am in bed unmotivated. i cant do my hair, i cant brush my teeth, clean my room, take a shower, im broken. the simple tasks like waking up in the morning take me days to do. school starts on monday and i cant do it. im getting help, but it isnt helping. i know in order for me to be okay, i have to put effort in. im trying, i really, really am. its not helping. my family thinks i have bipolar disorder, i think its a teenage thing. im in therapy. i hate it. she makes me close my eyes, but darkness makes me feel trapped. i have no appetite, i cant sleep at night, im currently writing this at 232 in the morning. i walked into the kitchen the oth